# 6 Ejemplo de MLP para clasificación

**MLP para MNIST:** $\quad$ entrada de $784$ dimensiones, capa oculta de $800$ ReLUs y salida logits para $10$ clases

In [1]:
import numpy as np; import datasets
ds = datasets.load_dataset("ylecun/mnist").with_format("numpy"); ds
X_train = ds['train'][:]['image'].astype(np.float32).reshape(-1, 28*28) / 255.
y_train = ds['train'][:]['label'].astype(np.uint8)
X_test = ds['test'][:]['image'].astype(np.float32).reshape(-1, 28*28) / 255.
y_test = ds['test'][:]['label'].astype(np.uint8)

In [2]:
import torch; from torch import nn
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
class NeuralNetwork(nn.Module):
    def __init__(self, input_shape, num_ReLUs=800):
        super().__init__()
        self.MLP = nn.Sequential(nn.Linear(input_shape, num_ReLUs), nn.ReLU(), 
            nn.Linear(num_ReLUs, num_ReLUs), nn.ReLU(), nn.Linear(num_ReLUs, 10))
    def forward(self, x):
        return self.MLP(x)
model = NeuralNetwork(X_train.shape[1]).to(device)

In [3]:
X = torch.Tensor(X_train).to(device); y = torch.Tensor(y_train).to(device, dtype=torch.long)
loss_fn = nn.CrossEntropyLoss(); optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
model.train(); epochs = 200
for t in range(epochs):
    pred = model(X); loss = loss_fn(pred, y); loss.backward(); optimizer.step(); optimizer.zero_grad()

In [4]:
X = torch.Tensor(X_test).to(device); pred = model(X).detach().cpu().numpy()
acc = (pred.argmax(1) == y_test).sum().item() / y_test.size
print(f'Precisión {acc:.2%}')

Precisión 97.61%



<p style="page-break-after:always;"></p>
